1. I git-cloned the MQP-Dashboard-Backend

2. I checked the files with 'ls' and saw 'pdm.lock' file. That means I need to install pdm.

3. Then I ran a pdm pytest and saw 'collected 22 items' message.

4. To exit I pressed 'Control+C' (NOT 'Cmd+C') twice.

1. Typed 'breakpoint()' after 'db = open_database(create_tables=True)'
* 'open_database(...)' creates the SQLite file + tables.
* 'open_database()' also decides which database to use (SQLite vs Postgres) by reading environment variables.
* 'db.disconnect()' closes the connection and after this, teardown may delete the file.

2. I run the pytest and see (Pdb).

3. I typed 'db.provider.pool.filename' and saw the following:
'/Users/di54rid/Desktop/MQP-Dashboard-Backend/test_db.sqlite'
* That shows where the local DB file is.

1. ChatGPT recommended me to add 'db_access.db_config.set_test_env()' to force 'open_database()' to use **SQLite**.
* He says: _Environment variables are read at DB-creation time, not retroactively._

2. instead of '@fixture' ChatGPT recommended '@pytest.fixture(scope="module", autouse=True)'
* ChatGPT: _So the DB is created once per module and all tests see it._
* ChatGPT also introduced '@pytest.fixture()', therefore, I needed to import pytest library.

3. Lastly, ChatGPT recommended 'db_access.db_config.reset_test_env()' to reset environment after cleanup

1. I completely changed the function 'test_display_user_by_pages()' in 'test_fetch_by_id.py' file

2. After running pytest, I get the error:
> assert len(result["jobs"]) == jobs_per_page E assert 0 == 20
* ChatGPT: _This means your test data doesn’t include any jobs for identity="testuser1"._

3. Thanks to the breakpoint, I access the file 'test_db.db' via sqlite using the code:
> sqlite3 /Users/di54rid/Desktop/MQSS-Hackathons/MQP-Database-Access/test_db.db

4. I get error because in the past, I modified the test for 'identity="testuser1"'. But in the database, 'circuit_job' does not have an identity column. Jobs belong to users via 'circuit_job.owner'

4. When I run pytest, I get the following error:
> FAILED tests/test_fetch_by_id.py::test_display_user_by_pages - psycopg2.OperationalError: connection to server on socket "/tmp/.s.PGSQL.5432" failed: No such file or directory
* ChatGPT: 'tests/utils.py::insert_random_data()' uses psycopg2 → Postgres, so it ignores your SQLite test DB and tries to connect to a local Postgres server (which you don’t have).

* '(Pdb) db.provider.dialect' # to see whether test uses SQL (or PostgreSQL)
* '(Pdb) db.provider.pool.filename' # to see the name and the directory of the test file

**Important Note:** My supervisor (Tobias) told me that conftest.py of Database should be similar as much as possible to that Dashboard-Backend. That means **No SQL requiring functions**.

Tobias: _For cleanliness and readability I would put all the functions for population into utils.py._

1. After havinga a relativaly long conversation with ChatGPT, I modified conftest.py of Database-Access accordingly. There are two important functions:
* 'def create_local_database(db_path: str = TEST_DB_PATH)'
* @pytest.fixture(scope="session", autouse=True); def setup_test_database():'

2. I have just learnt that 'open_database()' and 'db = Database()' are already defined globaly in the file '_database.py'

1. 'conftest.py' is responsible for creating and cleaning local databases for tests to run when Pytest is initialized.

2. Whether it will create for each test or whole session depends on 'scope'. In the MQSS-Hackathons, it was 'session' while in the MQS-Dashboard-Backend it is 'module'.

3. 'scope="module"' means **each test file gets itws own database** while 'scope="session"' means **one database for all tests**.

4. It seems better if I set 'scope="module"' for conftest.py in order to debug the code when pytest is ran.

**Very Important**: "No module named ..." errors. Add the package name to the dependencies of 'pyproject.toml' and then run 'pdm update'.

For instance, it currently looks like
> dependencies = ["pony>=0.7", "bcrypt>=4.0", "psycopg2-binary>=2.9.10", "Werkzeug==3.1.4", "python-ldap-test>=0.3.1", "pytest>=9.0.2"] 

That means if you run 'pdm update', you shouldn't be seeing.

1. I was seeing the error
> OperationalError: no such table: admin_announcements

Even though the table existed though null (I was able to see it via DB Browser).

**Note**: pgAdmin 4 only shows PostgreSQL, but pytest is using SQLite (as we confirmed before).

However, this does not guarantee that Pony should see it since it checks what should exist according to the ORM (Object–Relational Mapping) mapping.

**Note**: I don't know what ORM is.

Therefore, I deleted the line below from conftest.py

> db.disconnect()

I deleted the lines below from __init__.py:

* from . import config
* from . import login
* from . import tokens
* from . import jobs
* from . import resources
* from . import feedbacks
* from . import request_access

I wrote the following code instead:

In [1]:
from importlib import import_module

from ._database import open_database

_EXPOSE = {
    "open_database",
    "users",
    "jobs",
    "tokens",
    "resources",
    "feedbacks",
    "request_access",
    "db_config",
    "_database",
    "_constants",
}

__all__ = list(_EXPOSE)


def __getattr__(name: str):
    if name == "open_database":
        return open_database
    if name in _EXPOSE:
        return import_module(f"{__name__}.{name}")
    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")


ImportError: attempted relative import with no known parent package

To add packages to pdm, write the following code:

> pdm add cffi

I copy-pasted "@pytest.fixture()" definition functions from conftest.py of Backend-Dashboard to conftest.py of Database-Access.

1.  I got
    > E fixture 'empty_db' not found
    
    Therefore, I changed the name of 'def create_local_database()' to 'def empty_db()'

2. But this time, empty_db() is not empty, I confirmed by typing the following code after db is open:
    > with db_session:
        print(select(s.name for s in db.UserSecurityLevel)[:])

3. Now I have achieved to get rid of the error, my code looks like this (for now):

In [2]:
from pytest import fixture

@fixture

def empty_db(monkeypatch): # monkeypatch is pytest’s surgical tool for temporarily changing thing
    monkeypatch.setenv("ENVIRONMENT", "test")  # auto-restores after test

    db = open_database(create_tables=True)

    try:
        db.drop_all_tables(with_all_data=True)

        db.create_tables()
        
        yield db
    finally:
        db.disconnect()

def create_local_database(): 
    db = open_database(create_tables=True)
    breakpoint()

    db.disconnect()

To check if tables are created, I changed test_db() in the following way:

In [3]:
@fixture
def empty_db(monkeypatch):
    monkeypatch.setenv("ENVIRONMENT", "test")

    db = open_database(create_tables=True)
    try:
        db.drop_all_tables(with_all_data=True)

        assert db.entities, (
            "No Pony entities registered -> tables cannot be created. "
            "Ensure models are imported before create_tables()."
        )

        db.create_tables()

        with db_session:
            tables = db.select("""
                SELECT name
                FROM sqlite_master
                WHERE type='table' AND name NOT LIKE 'sqlite_%'
                ORDER BY name
            """)
            print("SQLite tables:", tables)
            assert tables, "SQLite DB exists but contains no tables."

        yield db
    finally:
        db.disconnect()

then I run

> pdm run pytest -q tests/test_users.py -s


Now, it looks like this:

In [ ]:
@fixture
def empty_db(monkeypatch):
    monkeypatch.setenv("ENVIRONMENT", "test")

    db = open_database(create_tables=True)
    try:
        db.drop_all_tables(with_all_data=True)

        assert db.entities, (
            "No Pony entities registered -> tables cannot be created. "
            "Ensure models are imported before create_tables()."
        )

        db.create_tables()

        with db_session:
            # instead of "db.select(...)"", we have "db.execute(...)" because it'd create a syntax error "select SELECT ..."
            cur = db.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type='table' AND name NOT LIKE 'sqlite_%'
                ORDER BY name
            """)
            tables = [row[0] for row in cur.fetchall()]
            print("SQLite tables:", tables)
            assert tables, "SQLite DB exists but contains no tables."

        yield db
    finally:
        db.disconnect()

and I get:

> (mqp-database-access-3.11) root@4ca6aabcd85b:/workspaces/MQP-Database-Access# pdm run pytest -q tests/test_users.py -s
SQLite tables: ['admin_announcements', 'budget', 'budget_resource', 'circuit_job', 'feedback', 'hamiltonian_job', 'pane_pointers', 'resource', 'resource_security_level', 'super_user_level', 'target_specification', 'time_slots', 'token', 'token_usage', 'user', 'user_group', 'user_groups_in_budgets', 'user_groups_in_time_slots', 'user_security_level', 'users_in_time_slots', 'users_in_user_groups']
.
======================================================================================= warnings summary =======================================================================================
.venv/lib/python3.11/site-packages/ldap_test/server.py:8
  /workspaces/MQP-Database-Access/.venv/lib/python3.11/site-packages/ldap_test/server.py:8: DeprecationWarning: The distutils package is deprecated and slated for removal in Python 3.12. Use setuptools or check PEP 632 for potential alternatives
    from distutils.spawn import find_executable

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 0.14s

What **empty_db()** actually guarantees

1. The right environment:

> ENVIRONMENT = "test"

2.  A fresh database:

> drop_all_tables()

> create_tables()

So the test starts from zero assumptions.

3. Isolation
Each test gets the same initial state.
Order no longer matters.

4. Automatic cleanup

> yield

> finally: db.disconnect()

**Pony ORM**

a Python Object–Relational Mapper — a library that lets you work with databases using Python objects instead of SQL.

**assert**

1. Basic form:
> assert condition

e.g.
> x = 5

> assert x > 0



2. For an error message:
> assert condition, "Something went wrong"


**Now, let's focus on the other problem.**

On the terminal, I run

> env | grep -E 'QUANTUM_DB|PGHOST|PGUSER|PGPASSWORD|PGDATABASE|PGPORT'

and saw

> QUANTUM_DB_TESTING=1

> QUANTUM_DB_FILENAME=test_db.db

**ChatGPT**: _**you have zero Postgres connection** settings in your environment. So psycopg2 has no host/user/password to use_

**ChatGPT**:
Your Postgres test (tests/utils.py) reads:
> QUANTUM_DB_USER

> QUANTUM_DB_PASS

> QUANTUM_DB_HOST

But your environment (and even your repo defaults) are setting these to empty strings:

pyproject.toml has QUANTUM_DB_USER=, QUANTUM_DB_PASS=, QUANTUM_DB_HOST= (empty)

and bqp_database_access/db_config.py literally sets them to ""


I have changed **pyproject.toml**  from

In [6]:
env = [
    "QUANTUM_DB_TESTING=TRUE",
    "QUANTUM_DB_FILENAME=test_db.sqlite",
    "QUANTUM_DB_USER=",
    "QUANTUM_DB_PASS=",
    "QUANTUM_DB_HOST=",
]

to

In [7]:
env = [
    "QUANTUM_DB_TESTING=TRUE",
    "QUANTUM_DB_FILENAME=test_db.sqlite",
    "QUANTUM_DB_USER=postgres",
    "QUANTUM_DB_PASS=example",
    "QUANTUM_DB_HOST=qdb",
]

I typed

> getent hosts qdb || echo "qdb not resolvable"

and got

> qdb not resolvable

**ChatGPT**:

* Postgres is not running inside your container (otherwise localhost:5432 wouldn’t be “connection refused”).

* The Postgres container/service named qdb is not reachable from your container (DNS fails).

* That means: a Postgres-dependent test cannot pass in this environment, full stop, unless you change the environment or change the test.

**ChatGPT**:

* tests/utils.insert_random_data(): **psycopg2/Postgres-only**

1. Choice A: Unit-test mode (SQLite/Pony)

✅ Test runs anywhere (including your container)
✅ No Postgres, no psycopg2
➡️ You must replace utils.insert_random_data(...) with Pony seeding.

This requires changing the test (or adding a local seeding helper). You don’t have to touch tests/utils.py if you don’t want to.

2. Choice B: Integration-test mode (Postgres/psycopg2)

✅ Tests your real Postgres path
➡️ You must run it where Postgres exists + qdb resolves (host machine / CI / docker-compose).
In this container, it will keep failing.